In [23]:
import os
import cv2
from matplotlib import pyplot as plt
import numpy as np
from IPython.display import Image, display

models_dir = 'models'
e = os.path.expanduser

In [46]:
# YuNet
yunet = cv2.FaceDetectorYN.create(
    "models/face_detection_yunet_2023mar.onnx",
    "",
    (320, 320),
    score_threshold=0.6
)
retina_net = cv2.dnn.readNetFromONNX("./models/retinaface-resnet50.onnx")

# Person detector (MobileNet SSD)
person_net = cv2.dnn.readNetFromCaffe(
    "models/mobilenet_iter_73000.prototxt",
    "models/mobilenet_iter_73000.caffemodel"
)

# Face fallback (ResNet SSD)
face_net = cv2.dnn.readNetFromCaffe(
    "models/res10_300x300_ssd_iter_140000.prototxt",
    "models/res10_300x300_ssd_iter_140000.caffemodel"
)

# COCO-like classes for MobileNet SSD
PERSON_CLASS_ID = 15

# ---------------------------
# Helpers
# ---------------------------

def resize_if_needed(image, max_dim=1024):
    h, w = image.shape[:2]
    scale = max_dim / max(h, w)
    if scale < 1:
        image = cv2.resize(image, None, fx=scale, fy=scale)
    return image

# ---------------------------
# YuNet
# ---------------------------

def detect_faces_yunet(image):
    h, w = image.shape[:2]
    yunet.setInputSize((w, h))
    _, faces = yunet.detect(image)
    
    if faces is None:
        return []
    return faces

# ---------------------------
# Person detection (cheap gate)
# ---------------------------

def detect_person(image):
    blob = cv2.dnn.blobFromImage(image, 0.007843, (300, 300), 127.5)
    person_net.setInput(blob)
    detections = person_net.forward()
    
    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        cls = int(detections[0, 0, i, 1])
        
        if conf > 0.5 and cls == PERSON_CLASS_ID:
            return True
    return False
def detect_faces_retinaface(image, conf_threshold=0.7):
    h, w = image.shape[:2]

    # ---------------------------
    # Preprocess
    # ---------------------------
    input_size = 640

    blob = cv2.dnn.blobFromImage(
        image,
        scalefactor=1.0,
        size=(input_size, input_size),
        mean=(104, 117, 123),
        swapRB=True,
        crop=False
    )

    retina_net.setInput(blob)
    outputs = retina_net.forward()

    # ⚠️ aqui depende do modelo
    # vamos assumir formato típico:
    # [num_detections, 15] (bbox + score + landmarks)

    detections = outputs.squeeze()

    faces = []

    for det in detections:
        score = det[4]

        if score < conf_threshold:
            continue

        # bbox normalizado (0–1)
        x1 = int(det[0] * w)
        y1 = int(det[1] * h)
        x2 = int(det[2] * w)
        y2 = int(det[3] * h)

        bw = x2 - x1
        bh = y2 - y1

        # landmarks
        landmarks = []
        for i in range(5):
            lx = int(det[5 + i*2] * w)
            ly = int(det[5 + i*2 + 1] * h)
            landmarks.extend([lx, ly])

        faces.append([
            x1, y1, bw, bh,
            float(score),
            *landmarks
        ])

    return np.array(faces, dtype=np.float32)
    
# ---------------------------
# Fallback face detector
# ---------------------------

def detect_faces_fallback(image, conf_threshold=0.6):
    h, w = image.shape[:2]

    # 🔴 GARANTIR formato correto
    image = np.ascontiguousarray(image)
    image = image.astype("uint8")

    # 🔴 blob correto (isso aqui é crítico)
    blob = cv2.dnn.blobFromImage(
        image,
        scalefactor=1.0,
        size=(300, 300),
        mean=(104.0, 177.0, 123.0),
        swapRB=False,
        crop=False
    )

    face_net.setInput(blob)
    detections = face_net.forward()

    faces = []

    # formato: [1, 1, N, 7]
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > conf_threshold:
            box = detections[0, 0, i, 3:7]

            x1 = int(box[0] * w)
            y1 = int(box[1] * h)
            x2 = int(box[2] * w)
            y2 = int(box[3] * h)

            # clamp (evita bug de coordenada negativa)
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(w, x2)
            y2 = min(h, y2)

            faces.append((x1, y1, x2 - x1, y2 - y1))

    return faces
    

In [39]:
def display_image(source, width=None, height=None):
    """
    Displays an image from a file path, cv2 array, or numpy array using Matplotlib.
    
    Args:
        source: str (path), np.ndarray (cv2/numpy array).
        width: Visual width in pixels (approximate via dpi).
        height: Visual height in pixels (approximate via dpi).
    """
    image = None

    # 1. Handle string path
    if isinstance(source, str):
        image = cv2.imread(source)
        if image is None:
            print(f"Error: Could not load image from {source}")
            return
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 2. Handle numpy/cv2 array
    elif isinstance(source, np.ndarray):
        # If it has 3 channels, OpenCV usually loads as BGR; convert to RGB
        if len(source.shape) == 3 and source.shape[2] == 3:
            image = cv2.cvtColor(source, cv2.COLOR_BGR2RGB)
        else:
            image = source
    else:
        print("Unsupported format. Use a file path (str) or a numpy array.")
        return

    # 3. Setup visual resize using figsize (inches = pixels / dpi)
    dpi = 100
    fig_w = width / dpi if width else None
    fig_h = height / dpi if height else None
    
    plt.figure(figsize=(fig_w if fig_w else 6, fig_h if fig_h else 4))
    
    if len(image.shape) == 2:
        plt.imshow(image, cmap='gray')
    else:
        plt.imshow(image)
        
    plt.axis('off')
    plt.show()



In [40]:
olivia_1 = e('~/Photos/2021/01/31/IMG_20210131_085157.jpg')
olivia_1_cv = cv2.imread(olivia_1)
olivia_2 = e('~/Photos/2021/01/31/IMG_20210131_085157.jpg')
olivia_2_cv = cv2.imread(olivia_2)
olivia_3 = e('~/Photos/2021/01/25/IMG_20210125_183051.jpg')
olivia_3_cv = cv2.imread(olivia_3)
olivia_4 = e('~/Photos/2021/01/22/IMG_20210122_183711.jpg')
olivia_4_cv = cv2.imread(olivia_4)

#marcelo_1 = e('~/Photos/2021/01/22/IMG_20210122_15136.jpg')
#marcelo_1_cv = cv2.imread(marcelo_1)

In [48]:
face = detect_faces_retinaface(olivia_1_cv)
#print(len(face_d),face_d)
#face = face_d[0][:4].astype(int)
print(face)
img2 = cv2.rectangle(olivia_1_cv, (face[0], face[2]), (face[1], face[3]), (0, 255, 0), 2)
display_image(img2)

IndexError: index 10 is out of bounds for axis 0 with size 10

In [6]:
import os
import cv2
import numpy as np

models_dir = "models"
nets = {}

for root, dirs, files in os.walk(models_dir):
    for f in files:
        path = os.path.join(root, f)
        name = os.path.relpath(path, models_dir)
        try:
            if f.lower().endswith(".onnx"):
                net = cv2.dnn.readNetFromONNX(path)
            elif f.lower().endswith(".caffemodel"):
                prototxt = None
                for candidate in files:
                    if candidate.lower().endswith(".prototxt"):
                        prototxt = os.path.join(root, candidate)
                        break
                if prototxt:
                    net = cv2.dnn.readNetFromCaffe(prototxt, path)
                else:
                    net = cv2.dnn.readNet(path)
            elif f.lower().endswith(".pb"):
                pbtxt = None
                for candidate in files:
                    if candidate.lower().endswith(".pbtxt"):
                        pbtxt = os.path.join(root, candidate)
                        break
                if pbtxt:
                    net = cv2.dnn.readNetFromTensorflow(path, pbtxt)
                else:
                    net = cv2.dnn.readNet(path)
            elif f.lower().endswith(".weights") or f.lower().endswith(".cfg"):
                cfg = None
                weights = None
                for candidate in files:
                    if candidate.lower().endswith(".cfg"):
                        cfg = os.path.join(root, candidate)
                    if candidate.lower().endswith(".weights"):
                        weights = os.path.join(root, candidate)
                if cfg and weights:
                    net = cv2.dnn.readNetFromDarknet(cfg, weights)
                else:
                    net = cv2.dnn.readNet(path)
            else:
                net = cv2.dnn.readNet(path)
            nets[name] = net
            print(f"Loaded {name}")
        except Exception as e:
            print(f"Failed to load {name}: {e}")

Loaded res10_300x300_ssd_iter_140000.caffemodel
Loaded deploy.prototxt
Failed to load BlazeFace-1.0: OpenCV(4.13.0) /io/opencv/modules/dnn/src/dnn_read.cpp:58: error: (-2:Unspecified error) Cannot determine an origin framework of files: models/BlazeFace-1.0 in function 'readNet'

Failed to load ArcFace-ResNet100-v1: OpenCV(4.13.0) /io/opencv/modules/dnn/src/dnn_read.cpp:58: error: (-2:Unspecified error) Cannot determine an origin framework of files: models/ArcFace-ResNet100-v1 in function 'readNet'

Failed to load Dlib-FaceLandmark-68: OpenCV(4.13.0) /io/opencv/modules/dnn/src/dnn_read.cpp:58: error: (-2:Unspecified error) Cannot determine an origin framework of files: models/Dlib-FaceLandmark-68 in function 'readNet'

Loaded mobilenet_iter_73000.caffemodel
Failed to load DeepFace-VGGFace2: OpenCV(4.13.0) /io/opencv/modules/dnn/src/dnn_read.cpp:58: error: (-2:Unspecified error) Cannot determine an origin framework of files: models/DeepFace-VGGFace2 in function 'readNet'

Loaded face_de

In [7]:
# Usage example: run a forward pass with the first loaded model
import numpy as np

if not nets:
    print("No nets were loaded. Check the models/ folder.")
else:
    name, net = next(iter(nets.items()))
    print("Using model:", name)
    # Create a dummy image blob (adjust size as needed per model)
    blob = cv2.dnn.blobFromImage(np.zeros((224,224,3), dtype=np.uint8), scalefactor=1.0, size=(224,224), mean=(0,0,0), swapRB=True, crop=False)
    net.setInput(blob)
    try:
        out = net.forward()
        try:
            print("Output shape:", out.shape)
        except Exception:
            print("Output (non-array):", type(out))
    except Exception as e:
        print("Forward pass failed:", e)

Using model: res10_300x300_ssd_iter_140000.caffemodel
Forward pass failed: OpenCV(4.13.0) /io/opencv/modules/dnn/src/layers/convolution_layer.cpp:369: error: (-215:Assertion failed) !blobs.empty() || inputs.size() > 1 in function 'getMemoryShapes'



[ERROR:0@0.700] global net_impl.cpp:1166 getLayerShapesRecursively OPENCV/DNN: [Convolution]:(conv0): getMemoryShapes() throws exception. inputs=1 outputs=0/1 blobs=0
[ERROR:0@0.700] global net_impl.cpp:1172 getLayerShapesRecursively     input[0] = [ 1 3 224 224 ]
[ERROR:0@0.700] global net_impl.cpp:1182 getLayerShapesRecursively Exception message: OpenCV(4.13.0) /io/opencv/modules/dnn/src/layers/convolution_layer.cpp:369: error: (-215:Assertion failed) !blobs.empty() || inputs.size() > 1 in function 'getMemoryShapes'

